In [4]:
import pandas as pd

df_a = pd.read_csv("../data/raw/hubspot_instance_a_contacts.csv")
df_b = pd.read_csv("../data/raw/hubspot_instance_b_contacts.csv")

In [5]:
a_rename = {
    'hs_record_id': 'lead_id',
    'email_address': 'email',
    'firstname': 'first_name',
    'lastname': 'last_name',
    'company': 'company',
    'job_title': 'job_title',
    'lead_source': 'source',
    'hs_lifecycle_stage': 'lifecycle_stage',
    'created_at': 'created_date',
    'mql_became_at': 'mql_date',
    'sql_became_at': 'sql_date',
    'opportunity_created_at': 'opportunity_date',
    'customer_at': 'customer_date',
    'owner_name': 'owner',
    'manual_entry_notes': 'notes',
    'last_modified_at': 'last_updated'
}

In [6]:
b_rename = {
    'contact_id': 'lead_id',
    'email': 'email',
    'first_name': 'first_name',
    'last_name': 'last_name',
    'company_name': 'company',
    'title': 'job_title',
    'original_source': 'source',
    'lifecycle_stage': 'lifecycle_stage',
    'create_date': 'created_date',
    'mql_date': 'mql_date',
    'sal_date': 'sql_date',   # IMPORTANT mapping
    'opportunity_date': 'opportunity_date',
    'customer_date': 'customer_date',
    'sales_owner': 'owner',
    'notes': 'notes',
    'last_touch_date': 'last_updated'
}

In [7]:
df_a_std = df_a.rename(columns=a_rename)
df_b_std = df_b.rename(columns=b_rename)

In [8]:
df_a_std['source_system'] = 'hubspot_A'
df_b_std['source_system'] = 'hubspot_B'

In [9]:
df_a_std['email'] = df_a_std['email'].str.lower().str.strip()
df_b_std['email'] = df_b_std['email'].str.lower().str.strip()

In [11]:
df_a_std['lifecycle_stage'].unique()

array(['Lead', 'Opportunity', 'SQL', 'MQL ', 'mql', 'Sales Qualified',
       'Customer', 'MQL', 'SQL ', 'Marketing Qualified Lead'],
      dtype=object)

In [12]:
df_b_std['lifecycle_stage'].unique()

array(['Customer', 'SAL', 'Subscriber', 'MQL', 'Marketing Qualified',
       'lead', 'Oppty', 'Closed Won', 'sql'], dtype=object)

In [13]:
stage_mapping = {
    # Lead
    'lead': 'Lead',
    'Lead': 'Lead',

    # MQL
    'MQL': 'MQL',
    'mql': 'MQL',
    'Marketing Qualified': 'MQL',
    'Marketing Qualified Lead': 'MQL',

    # SQL
    'SQL': 'SQL',
    'sql': 'SQL',
    'Sales Qualified': 'SQL',
    'SAL': 'SQL',

    # Opportunity
    'Opportunity': 'Opportunity',
    'Oppty': 'Opportunity',

    # Customer
    'Customer': 'Customer',
    'Closed Won': 'Customer',

    # Subscriber (keep separate or map to Lead)
    'Subscriber': 'Lead'
}

In [14]:
df_a_std['lifecycle_stage'] = df_a_std['lifecycle_stage'].map(stage_mapping)
df_b_std['lifecycle_stage'] = df_b_std['lifecycle_stage'].map(stage_mapping)

In [15]:
df_a_std['lifecycle_stage'].value_counts()

Lead           9
Opportunity    9
SQL            6
MQL            3
Customer       2
Name: lifecycle_stage, dtype: int64

In [16]:
df_b_std['lifecycle_stage'].value_counts()

Lead           12
MQL             9
SQL             7
Customer        5
Opportunity     3
Name: lifecycle_stage, dtype: int64

In [17]:
date_cols = ['created_date', 'mql_date', 'sql_date', 'opportunity_date', 'customer_date']

for col in date_cols:
    df_a_std[col] = pd.to_datetime(df_a_std[col], errors='coerce')
    df_b_std[col] = pd.to_datetime(df_b_std[col], errors='coerce')

In [18]:
df_a_std.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 36 entries, 0 to 35
Data columns (total 17 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   lead_id           36 non-null     object        
 1   email             36 non-null     object        
 2   first_name        36 non-null     object        
 3   last_name         36 non-null     object        
 4   company           36 non-null     object        
 5   job_title         36 non-null     object        
 6   source            36 non-null     object        
 7   lifecycle_stage   29 non-null     object        
 8   created_date      36 non-null     datetime64[ns]
 9   mql_date          24 non-null     datetime64[ns]
 10  sql_date          19 non-null     datetime64[ns]
 11  opportunity_date  11 non-null     datetime64[ns]
 12  customer_date     2 non-null      datetime64[ns]
 13  owner             36 non-null     object        
 14  notes             33 non-nul

In [19]:
df_b_std.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 36 entries, 0 to 35
Data columns (total 17 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   lead_id           36 non-null     object        
 1   email             36 non-null     object        
 2   first_name        36 non-null     object        
 3   last_name         36 non-null     object        
 4   company           36 non-null     object        
 5   job_title         36 non-null     object        
 6   source            36 non-null     object        
 7   lifecycle_stage   36 non-null     object        
 8   created_date      36 non-null     datetime64[ns]
 9   mql_date          21 non-null     datetime64[ns]
 10  sql_date          15 non-null     datetime64[ns]
 11  opportunity_date  8 non-null      datetime64[ns]
 12  customer_date     5 non-null      datetime64[ns]
 13  owner             36 non-null     object        
 14  notes             31 non-nul

In [20]:
common_cols = [
    'lead_id', 'email', 'first_name', 'last_name', 'company',
    'job_title', 'source', 'lifecycle_stage',
    'created_date', 'mql_date', 'sql_date', 'opportunity_date', 'customer_date',
    'owner', 'notes', 'last_updated', 'source_system'
]

df_a_std = df_a_std[common_cols]
df_b_std = df_b_std[common_cols]

In [21]:
df_a_std['lifecycle_stage'] = df_a_std['lifecycle_stage'].str.strip()
df_b_std['lifecycle_stage'] = df_b_std['lifecycle_stage'].str.strip()

In [22]:
df_a_std['lifecycle_stage'] = df_a_std['lifecycle_stage'].map(stage_mapping)
df_b_std['lifecycle_stage'] = df_b_std['lifecycle_stage'].map(stage_mapping)

In [23]:
df_a_std['lifecycle_stage'].isna().sum()
df_b_std['lifecycle_stage'].isna().sum()

0

In [24]:
df_a_std.to_csv("../data/interim/hubspot_a_standardized.csv", index=False)
df_b_std.to_csv("../data/interim/hubspot_b_standardized.csv", index=False)